# Platform ops health — notebook

Operational integrity report: publish-queue state, failed targets with
error signatures, follower-series anomalies (the corrupted-snapshot guard),
analytics-sync freshness per account, and a 30-day content scoreboard
(engagement vs platform median → real outliers to replicate).

Run: **Run All** or `run_notebook_report("ops_health")`. Safe to run anytime —
read-only queries.

In [ ]:
# papermill-injected parameters (tagged "parameters")
lookback_days = 7
scoreboard_days = 30
output_dir = "/notebooks/output"
run_id = "manual"
manifest_path = ""
send_hint = False

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime, timedelta, UTC

sys.path.insert(0, "/app")

import pandas as pd
from sqlalchemy import text
from app.db.session import async_session_maker

OUT = Path(output_dir)
OUT.mkdir(parents=True, exist_ok=True)
now = datetime.now(UTC)
print("ops health @", now.isoformat())

In [ ]:
async def _q(sql, **kw):
    async with async_session_maker() as db:
        return (await db.execute(text(sql), kw)).mappings().all()

# ── 1. Publish queue state ───────────────────────────────────────────────
queue_rows = await _q('''
    SELECT sa.platform, pq.status, count(*) n
    FROM publish_queue pq JOIN social_accounts sa ON sa.id=pq.social_account_id
    GROUP BY 1,2 ORDER BY 1,2''')
stale = await _q('''
    SELECT sa.platform, LEFT(p.content_text,60) title, pq.locked_at
    FROM publish_queue pq
    JOIN posts p ON p.id=pq.post_id
    JOIN social_accounts sa ON sa.id=pq.social_account_id
    WHERE pq.status='processing' AND pq.locked_at < :cut''',
    cut=now - timedelta(minutes=15))
failed_targets = await _q('''
    SELECT sa.platform, LEFT(p.content_text,55) title,
           LEFT(pt.error_message,110) err, pt.status,
           CAST(COALESCE(pt.published_at, p.created_at) AS date) d
    FROM post_targets pt
    JOIN posts p ON p.id=pt.post_id
    JOIN social_accounts sa ON sa.id=pt.social_account_id
    WHERE pt.status IN ('failed','skipped')
      AND p.created_at > now() - interval ':d days'
    ORDER BY d DESC'''.replace(':d', str(lookback_days)))

In [ ]:
# ── 2. Follower-series integrity ─────────────────────────────────────────
# A snapshot is suspicious when |delta| vs the previous point exceeds the
# plausible bound: >50% move AND >25 absolute for accounts <1k followers,
# or >20% for larger ones — the signature of two interleaved series.
followers = await _q('''
    SELECT sa.platform, sa.username, fs.followers, fs.captured_at,
           lag(fs.followers) OVER w AS prev,
           first_value(fs.followers) OVER w AS first_in_series
    FROM follower_snapshots fs
    JOIN social_accounts sa ON sa.id=fs.social_account_id
    WHERE fs.captured_at > now() - interval '14 days'
    WINDOW w AS (PARTITION BY fs.social_account_id ORDER BY fs.captured_at)
    ORDER BY sa.platform, fs.captured_at''')
anoms = []
for r in followers:
    if r["prev"] is None or r["prev"] == 0:
        # prev==0 is series initialisation (first real count after a 0
        # record), not a corrupted jump — skip it.
        continue
    delta = r["followers"] - r["prev"]
    bound = max(25, abs(r["prev"]) * (0.5 if r["prev"] < 1000 else 0.2))
    if abs(delta) > bound:
        anoms.append({**r, "delta": delta})

# ── 3. Analytics freshness — latest snapshot age per account ─────────────
freshness = await _q('''
    SELECT sa.platform, sa.username,
           max(fs.captured_at) AS last_follower,
           (SELECT max(pas.captured_at) FROM post_analytics_snapshots pas
             WHERE pas.social_account_id=sa.id) AS last_post_metrics,
           sa.status
    FROM social_accounts sa
    LEFT JOIN follower_snapshots fs ON fs.social_account_id=sa.id
    WHERE sa.status='active'
    GROUP BY sa.id, sa.platform, sa.username, sa.status
    ORDER BY sa.platform''')
stale_sync = [r for r in freshness
              if r["last_post_metrics"] and (now - r["last_post_metrics"].replace(tzinfo=UTC if r["last_post_metrics"].tzinfo is None else r["last_post_metrics"].tzinfo)) > timedelta(hours=2)
              or r["last_post_metrics"] is None]

In [ ]:
# ── 4. Content scoreboard (30d) — engagement vs platform median ─────────
board = await _q('''
    SELECT sa.platform, LEFT(p.content_text,60) title,
           pas.impressions, pas.engagement, pas.clicks,
           CAST(pas.captured_at AS date) d
    FROM post_analytics_snapshots pas
    JOIN social_accounts sa ON sa.id=pas.social_account_id
    LEFT JOIN posts p ON p.id=pas.post_id
    WHERE pas.captured_at > now() - interval ':d days'
      AND pas.engagement > 0
    ORDER BY pas.captured_at DESC'''.replace(':d', str(scoreboard_days)))
df = pd.DataFrame([dict(r) for r in board])
if not df.empty:
    df = df.sort_values("d").drop_duplicates(
        subset=["platform","title"], keep="last")
    med = df.groupby("platform")["engagement"].median()
    df["x_median"] = (df["engagement"] /
                      df["platform"].map(med).clip(lower=1)).round(1)
    df = df.sort_values("x_median", ascending=False)
    outliers = df[df["x_median"] >= 3].head(10)
else:
    outliers = df

In [ ]:
# ── Render text report ───────────────────────────────────────────────────
L = []
L.append(f"Platform ops health — {now:%Y-%m-%d %H:%M} UTC")
L.append("")
L.append("PUBLISH QUEUE")
if queue_rows:
    for r in queue_rows:
        L.append(f"  {r['platform']:<10} {r['status']:<10} {r['n']}")
else:
    L.append("  (empty)")
L.append("")
L.append(f"STALE LOCKS (>15min processing): {len(stale)}")
for r in stale:
    L.append(f"  {r['platform']} | {r['title']} | locked {r['locked_at']}")
L.append("")
L.append(f"FAILED/SKIPPED TARGETS ({lookback_days}d): {len(failed_targets)}")
for r in failed_targets:
    L.append(f"  {r['d']} {r['platform']:<9} {r['status']:<8} {r['title']}")
    L.append(f"      → {r['err']}")
L.append("")
L.append(f"FOLLOWER ANOMALIES (14d): {len(anoms)}")
for r in anoms[:15]:
    L.append(f"  {r['platform']:<10} @{r['username']}: {r['prev']} → {r['followers']} ({r['delta']:+d}) @ {r['captured_at']:%m-%d %H:%M}")
L.append("")
L.append("SYNC FRESHNESS (active accounts)")
for r in freshness:
    lm = r["last_post_metrics"]
    age = f"{(now - lm).total_seconds()/3600:.1f}h" if lm else "never"
    flag = " ⚠️" if (lm is None or (now - lm) > timedelta(hours=2)) else ""
    L.append(f"  {r['platform']:<10} @{r['username']:<22} last metrics: {age}{flag}")
L.append("")
L.append(f"CONTENT SCOREBOARD ({scoreboard_days}d) — engagement × platform median")
if not df.empty:
    for _, r in df.head(15).iterrows():
        star = " ★" if r["x_median"] >= 3 else ""
        L.append(f"  {r['platform']:<9} {r['engagement']:>4} eng {r['x_median']:>5}×{star}  {r['title']}")
    L.append("")
    L.append(f"Outliers (≥3× median): {len(outliers)} — replicate their format/topic.")
else:
    L.append("  no engagement data in window")

txt = "\n".join(L)
txt_file = OUT / f"ops_health-{run_id}.txt"
txt_file.write_text(txt)
print(txt[:2000])

In [ ]:
# ── Render HTML + manifest (same contract as daily_strategy_brief) ───────
import html as _html
rows = "".join(
    f"<tr><td>{_html.escape(str(r['platform']))}</td>"
    f"<td>{_html.escape(r['title'])}</td>"
    f"<td><code>{_html.escape(str(r['err']))}</code></td></tr>"
    for r in failed_targets)
board_rows = "" if df.empty else "".join(
    f"<tr><td>{r['platform']}</td><td>{int(r['engagement'])}</td>"
    f"<td><b>{r['x_median']}×</b></td><td>{_html.escape(str(r['title']))}</td></tr>"
    for _, r in df.head(15).iterrows())
anom_rows = "".join(
    f"<tr><td>{r['platform']}</td><td>@{r['username']}</td>"
    f"<td>{r['prev']} → {r['followers']} ({r['delta']:+d})</td></tr>"
    for r in anoms[:15])
fresh_rows = "".join(
    f"<tr><td>{r['platform']}</td><td>@{r['username']}</td>"
    f"<td>{'never' if r['last_post_metrics'] is None else r['last_post_metrics'].strftime('%m-%d %H:%M')}</td></tr>"
    for r in freshness)

html_doc = f'''<!doctype html><html><body style="font-family:sans-serif;max-width:900px;margin:auto;padding:20px;color:#1a1a1a">
<h2>Platform ops health — {now:%Y-%m-%d}</h2>
<h3>Publish queue</h3><ul>{"".join(f"<li>{r['platform']} · {r['status']} · <b>{r['n']}</b></li>" for r in queue_rows)}</ul>
<p>Stale locks: <b>{len(stale)}</b></p>
<h3>Failed / skipped targets ({lookback_days}d)</h3>
<table border=1 cellpadding=5 style="border-collapse:collapse;width:100%"><tr><th>platform</th><th>post</th><th>error</th></tr>{rows or "<tr><td colspan=3>none</td></tr>"}</table>
<h3>Follower-series anomalies</h3>
<table border=1 cellpadding=5 style="border-collapse:collapse;width:100%"><tr><th>platform</th><th>account</th><th>jump</th></tr>{anom_rows or "<tr><td colspan=3>none</td></tr>"}</table>
<h3>Sync freshness</h3>
<table border=1 cellpadding=5 style="border-collapse:collapse;width:100%"><tr><th>platform</th><th>account</th><th>last metrics (UTC)</th></tr>{fresh_rows}</table>
<h3>Content scoreboard ({scoreboard_days}d)</h3>
<table border=1 cellpadding=5 style="border-collapse:collapse;width:100%"><tr><th>platform</th><th>engagement</th><th>×median</th><th>post</th></tr>{board_rows or "<tr><td colspan=4>no data</td></tr>"}</table>
<pre style="background:#f6f8fa;padding:12px;border-radius:8px;white-space:pre-wrap">{_html.escape(txt)}</pre>
</body></html>'''
html_file = OUT / f"ops_health-{run_id}.html"
html_file.write_text(html_doc)

subject = (f"[SocialAuto] Ops health {now:%Y-%m-%d} — "
           f"{len(failed_targets)} failed · {len(anoms)} anomalies · {len(stale)} stale")
m_path = Path(manifest_path) if manifest_path else OUT / f"ops_health-{run_id}.manifest.json"
m_path.write_text(json.dumps({"reports": [{
    "subject": subject,
    "html_file": str(html_file),
    "text_file": str(txt_file),
    "attachments": [],
}]}, indent=2))
print("manifest →", m_path)
print(subject)